# 08 - NN_live and NN_axi: results and analysis

Two classifiers run on the board, built from the same recorded data (paper Sec. VII–VIII):

- **NN_live**, the streaming classifier: it classifies a 576-sample window of the boxcar stream as the samples arrive and
  decides 16.3 ns after the last one. It writes a per-shot record: shot number, the four first-layer sums, the logit, the
  sigmoid output and the decision.
- **NN_axi**, the conventional buffered hls4ml classifier: it reads the 770-sample window from QICK's buffer and writes a
  probability to BRAM for the host.

This notebook:
1. **Reproduces the paper's fidelities** from the v1.43 campaign data:
   - NN_live on all 100,000 test traces on both boards;
   - NN_axi on 20,000;
   - each with confidence intervals and the two error rates.
2. **Compares** board A with board B, and NN_live with NN_axi on the same shots (exact McNemar tests).
3. **Checks NN_live bit for bit** on its own recorded input:
   - first-layer sums against the integer weights;
   - decision against the kernel's rule;
   - sigmoid output against the fixed-point model.
4. **Optionally captures new shots** on a board and runs the same checks live.

**You need:** the campaign data in `campaign_v143/final/live/` and `campaign_v143/final/axi/` (raw captures listed in
`DATA_MANIFEST.csv`), the deployed model `models/model_signed_domain_seed20260901_w576_box.h5`, and for the sigmoid check, hls4ml
and a C++ compiler (see notebook 04). Part 4 also needs a board (notebooks 01 and 05).

In [ ]:
from pathlib import Path

DATA_DIR = Path("..")
CAMPAIGN = DATA_DIR / "campaign_v143" / "final"
LIVE_MODEL = DATA_DIR / "models/model_signed_domain_seed20260901_w576_box.h5"
CAL_ROWS = (1000, 1600)        # domain-map rows: used to fit the training data's map, so never scored
CHECK_SIGMOID = True           # compile the fixed-point model and compare NN_live's sigmoid output (needs a C++ compiler)
MSYS_BIN = Path(r"C:\msys64\ucrt64\bin")
WINDOW = 576

In [ ]:
import glob
import os
import re

os.environ.update(KERAS_BACKEND="tensorflow", TF_USE_LEGACY_KERAS="1", TF_CPP_MIN_LOG_LEVEL="3")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binomtest

Z = 1.959964


def fstats(pred, y):
    # Fidelity 1 - [P(1|0) + P(0|1)]/2 with a normal-approximation 95% CI, as in the paper.
    pred, y = np.asarray(pred).astype(int), np.asarray(y).astype(int)
    g, e = y == 0, y == 1
    p10, p01 = np.mean(pred[g] != 0), np.mean(pred[e] != 1)
    F = 1 - 0.5 * (p10 + p01)
    se = 0.5 * np.sqrt(p10 * (1 - p10) / g.sum() + p01 * (1 - p01) / e.sum())
    return dict(n=len(y), fid=100 * F, lo=100 * (F - Z * se), hi=100 * (F + Z * se), p10=100 * p10, p01=100 * p01)


def paired(pa, pb, y):
    # Same shots, two classifiers: exact McNemar test and the fidelity difference with its paired 95% CI.
    y = np.asarray(y).astype(int)
    oka, okb = np.asarray(pa).astype(int) == y, np.asarray(pb).astype(int) == y
    b, c = int((oka & ~okb).sum()), int((okb & ~oka).sum())
    d, var = 0.0, 0.0
    for cl in (0, 1):
        m = y == cl
        dd = oka[m].astype(int) - okb[m].astype(int)
        d += 0.5 * dd.mean(); var += 0.25 * dd.var() / m.sum()
    return dict(a_only=b, b_only=c, p=binomtest(b, b + c).pvalue if b + c else 1.0, dF=100 * d,
                lo=100 * (d - Z * np.sqrt(var)), hi=100 * (d + Z * np.sqrt(var)))


def show(name, s):
    print(f"{name:34s} {s['fid']:.2f}% [{s['lo']:.2f}, {s['hi']:.2f}]  P(1|0) {s['p10']:.2f}%  P(0|1) {s['p01']:.2f}%  n = {s['n']:,}")


def show_pair(name, r, a="A", b="B"):
    print(f"{name:34s} {a} only right {r['a_only']}, {b} only right {r['b_only']}, exact p = {r['p']:.3g}; "
          f"difference {r['dF']:+.3f} [{r['lo']:+.3f}, {r['hi']:+.3f}] points")


def sext(v, bits):
    v = np.asarray(v).astype(np.int64) & ((1 << bits) - 1)
    return np.where(v >= 1 << (bits - 1), v - (1 << bits), v)

## 1. NN_live on the full test split

Every trace was replayed on both boards. **Run 1:** board A scored rows 0–49,999 and board B rows 50,000–99,999. **Run 2:**
the halves swapped. The headline set is run 1, minus the 600 domain-map rows. An NN_live record has:
- word 0: shot number;
- words 1–4: first-layer sums;
- word 5: logit;
- word 6: sigmoid output;
- word 7: decision (bit 0).

In [ ]:
def load_live(board, run):
    fs = sorted(glob.glob(str(CAMPAIGN / "live" / f"{board}_run{run}_*.npz")),
                key=lambda f: int(re.findall(r"_(\d+)_\d+\.npz", f)[0]))
    parts = [np.load(f) for f in fs]
    d = {k: np.concatenate([p[k] for p in parts]) for k in ("rows", "y", "live_rec", "live_words", "dac_clipped")}
    o = np.argsort(d["rows"])
    return {k: v[o] for k, v in d.items()}


L = {(b, r): load_live(b, r) for b in "AB" for r in (1, 2)}
def join(a, b):          # combine two halves in row order (the windows stay in L, to save memory)
    o = np.argsort(np.concatenate([a["rows"], b["rows"]]))
    return {k: np.concatenate([a[k], b[k]])[o] for k in ("rows", "y", "live_rec", "dac_clipped")}


A, B, H = join(L["A", 1], L["A", 2]), join(L["B", 2], L["B", 1]), join(L["A", 1], L["B", 1])
dec = lambda d: (d["live_rec"][:, 7] & 1).astype(int)
cal = ~((H["rows"] >= CAL_ROWS[0]) & (H["rows"] < CAL_ROWS[1]))

show("NN_live headline (run 1)", fstats(dec(H)[cal], H["y"][cal]))
show("  all 100,000 rows", fstats(dec(H), H["y"]))
show("  every trace on board A", fstats(dec(A)[cal], A["y"][cal]))
show("  every trace on board B", fstats(dec(B)[cal], B["y"][cal]))
show_pair("  board A vs board B, same traces", paired(dec(A)[cal], dec(B)[cal], A["y"][cal]))
both_wrong = np.mean((dec(A) != A["y"]) & (dec(B) != A["y"]))
either = np.mean((dec(A) != A["y"]) | (dec(B) != A["y"]))
print(f"  decisions agree between boards on {100 * np.mean(dec(A)[cal] == dec(B)[cal]):.2f}% of shots; "
      f"wrong on both {100 * both_wrong:.2f}%, on at least one {100 * either:.2f}%")

Most errors are wrong on **both** boards: they belong to the trace (a decayed |1⟩, a thermally excited |0⟩), not to the
replay. The shots wrong on only one board measure the replay channel's own noise.

In [ ]:
logit = lambda d: sext(d["live_rec"][:, 5] & 0xFFFF, 16) / 1024.0
lg, yy = logit(H)[cal], H["y"][cal]
plt.figure(figsize=(7, 3))
bins = np.linspace(np.percentile(lg, 0.2), np.percentile(lg, 99.8), 120)
plt.hist(lg[yy == 0], bins, alpha=0.6, label="|0⟩"); plt.hist(lg[yy == 1], bins, alpha=0.6, label="|1⟩")
plt.axvline(1 / 64, color="k", lw=0.8, ls="--", label="threshold 1/64")
plt.yscale("log"); plt.xlabel("NN_live logit (Q.10)"); plt.ylabel("shots"); plt.legend(); plt.tight_layout(); plt.show()

## 2. NN_axi, and NN_live on the same shots

NN_axi ran on the first 20,000 test traces on each board. The same runs also recorded NN_live's decision on every shot,
so the two classifiers can be compared on identical replays.

In [ ]:
def load_axi(board):
    fs = sorted(glob.glob(str(CAMPAIGN / "axi" / f"{board}_axi_*.npz")), key=lambda f: int(re.findall(r"_axi_(\d+)\.npz", f)[0]))
    parts = [np.load(f, allow_pickle=True) for f in fs]
    d = dict(rows=np.concatenate([p["idx"] for p in parts]), p=np.concatenate([p["probs"] for p in parts]),
             y=np.concatenate([p["y"] for p in parts]).astype(int), rec=np.concatenate([p["live_rec"] for p in parts]))
    o = np.argsort(d["rows"])
    return {k: v[o] for k, v in d.items()}


AX = {b: load_axi(b) for b in "AB"}
m = ~((AX["A"]["rows"] >= CAL_ROWS[0]) & (AX["A"]["rows"] < CAL_ROWS[1]))
for b in "AB":
    d = AX[b]
    axi_dec, live_dec = (d["p"] > 0.5).astype(int), (d["rec"][:, 7] & 1).astype(int)
    show(f"NN_axi, board {b}", fstats(axi_dec[m], d["y"][m]))
    show(f"NN_live, same shots, board {b}", fstats(live_dec[m], d["y"][m]))
    show_pair(f"  NN_axi vs NN_live, board {b}", paired(axi_dec[m], live_dec[m], d["y"][m]), "NN_axi", "NN_live")
show_pair("NN_axi, board A vs board B", paired((AX["A"]["p"] > 0.5)[m], (AX["B"]["p"] > 0.5)[m], AX["A"]["y"][m]))
pool_d = np.concatenate([(AX[b]["p"] > 0.5)[m] for b in "AB"]); pool_y = np.concatenate([AX[b]["y"][m] for b in "AB"])
show("NN_axi, both boards pooled", fstats(pool_d, pool_y))

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
d = AX["A"]
ax[0].hist(d["p"][m & (d["y"] == 0)], 100, alpha=0.6, label="|0⟩"); ax[0].hist(d["p"][m & (d["y"] == 1)], 100, alpha=0.6, label="|1⟩")
ax[0].set_yscale("log"); ax[0].set_xlabel("NN_axi probability (board A)"); ax[0].legend()
both = np.c_[(d["p"] > 0.5).astype(int), (d["rec"][:, 7] & 1).astype(int)][m]
right = both == d["y"][m][:, None]
labels = ["both right", "NN_axi only", "NN_live only", "both wrong"]
counts = [np.sum(right[:, 0] & right[:, 1]), np.sum(right[:, 0] & ~right[:, 1]), np.sum(~right[:, 0] & right[:, 1]), np.sum(~right[:, 0] & ~right[:, 1])]
ax[1].bar(labels[1:], counts[1:]); ax[1].set_ylabel("shots (board A)"); ax[1].set_title(f"{counts[0]:,} shots both right")
plt.tight_layout(); plt.show()

The two networks are trained on different inputs: NN_axi on the FIR path's full 770-sample window, NN_live on the boxcar
path's first 576 samples. So they are expected to differ by a little in fidelity and to disagree on a few shots. The
McNemar lines say whether a difference is real.

## 3. NN_live, bit for bit

Every campaign record also holds the window NN_live actually classified (`live_words`: 576 words of packed signed 16-bit
I and Q). On that input:
1. **First-layer sums.** Each of the four sums must equal $W x$ exactly, where $W$ holds the integer ternary weights.
   The firmware's integer weights are the signs of the model's quantized kernel; each unit's scale is folded into the
   next stage.
2. **Decision rule.** The decision bit must equal (logit ≥ 1/64).
3. **Sigmoid output.** It must equal the fixed-point model's output on the same window.

Any mismatch would point at a timing fault inside NN_live, which the paper rules out with exactly this check.

In [ ]:
from qkeras.utils import load_qmodel

model = load_qmodel(str(LIVE_MODEL), compile=False)
L1 = model.get_layer("q_dense_1")
kq = np.array(L1.kernel_quantizer_internal(L1.get_weights()[0]))
W = np.sign(kq).T.astype(np.int64)                         # (4, 1152) in {-1, 0, +1}
bq = np.array(L1.bias_quantizer_internal(L1.get_weights()[1]))
b = np.sign(bq).astype(np.int64)
print("first-layer weights", W.shape, "values", np.unique(W), "bias", b)


def window_inputs(words):
    words = np.asarray(words).astype(np.uint32)
    x = np.empty((len(words), 2 * words.shape[1]), dtype=np.int64)
    x[:, 0::2] = (words & 0xFFFF).astype(np.uint16).view(np.int16)
    x[:, 1::2] = (words >> 16).astype(np.uint16).view(np.int16)
    return x


def check_sums_and_rule(words, rec):
    x, rec = window_inputs(words), np.asarray(rec).astype(np.int64)
    sums = sext(rec[:, 1:5], 27)                            # 27-bit accumulators
    ok_sums = (sums == x @ W.T + b).all(1)
    lg = sext(rec[:, 5] & 0xFFFF, 16) / 1024.0
    ok_rule = (rec[:, 7] & 1) == (lg >= 1 / 64)
    return ok_sums, ok_rule, x


n_tot = n_sums = n_rule = 0
for key, d in L.items():
    ok_s, ok_r, _ = check_sums_and_rule(d["live_words"], d["live_rec"])
    n_tot += len(ok_s); n_sums += ok_s.sum(); n_rule += ok_r.sum()
print(f"first-layer sums exact: {n_sums:,}/{n_tot:,} shots;  decision rule: {n_rule:,}/{n_tot:,}")

**Sigmoid output vs the fixed-point model.** This compiles the hls4ml C++ model (as in notebook 04) and runs it on every
recorded window. NN_live's sigmoid output is in word 6, in units of 1/1024. On Windows the compile needs MSYS2's g++.

In [ ]:
if CHECK_SIGMOID:
    import hls4ml

    if os.name == "nt":
        import subprocess
        from hls4ml.backends.fpga.fpga_backend import FPGABackend
        os.add_dll_directory(str(MSYS_BIN))

        def _compile_with_msys(self, mdl):
            odir = Path(mdl.config.get_output_dir()); sh = odir / "build_lib.sh"
            sh.write_text(sh.read_text().replace('&& pwd)"', '&& pwd -W)"'), newline="\n")
            r = subprocess.run(["bash", "build_lib.sh"], cwd=odir, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               env=dict(os.environ, PATH=str(MSYS_BIN) + os.pathsep + os.environ["PATH"]))
            if r.returncode != 0:
                print(r.stdout); raise RuntimeError("build_lib.sh failed")
            return "{}/firmware/{}-{}.so".format(mdl.config.get_output_dir(), mdl.config.get_project_name(),
                                                 mdl.config.get_config_value("Stamp"))
        FPGABackend.compile = _compile_with_msys

    cfg = hls4ml.utils.config_from_keras_model(model, granularity="name")
    cfg["Model"].update(Precision="ap_fixed<16,6>", ReuseFactor=1, Strategy="Resource")
    LN = cfg["LayerName"]
    def res(layer, p):
        if layer in LN:
            if not isinstance(LN[layer].get("Precision"), dict): LN[layer]["Precision"] = {}
            LN[layer]["Precision"]["result"] = p
    for nm in ("input_1", "input_1_input", "q_dense_1_input"):
        if nm in LN: LN[nm]["Precision"] = "ap_fixed<32,20>"
    res("q_dense_1", "ap_fixed<48,32>"); LN["q_dense_1"]["accum_t"] = "ap_fixed<48,32>"
    res("q_output", "ap_fixed<32,16>"); LN["q_output"]["accum_t"] = "ap_fixed<32,16>"
    for nm in ("batchnorm1", "relu1"): res(nm, "ap_fixed<32,16>")
    res("sigmoid", "ap_fixed<18,8>")
    for nm in list(LN):
        if "alpha" in nm or "linear" in nm: res(nm, "ap_fixed<24,10>")
    res("q_dense_1_linear", "ap_fixed<48,32>"); res("q_output_linear", "ap_fixed<32,16>")
    if "batchnorm1" in LN:
        LN["batchnorm1"]["Precision"].update(scale="ap_fixed<40,8>", bias="ap_fixed<40,8>")
    hm = hls4ml.converters.convert_from_keras_model(model=model, hls_config=cfg, output_dir="outputs/nn_live_fixed_point",
                                                    part="xczu48dr-ffvg1517-2-e", io_type="io_parallel", clock_period=10.0,
                                                    backend="Vivado", project_name="NN")
    hm.compile()

    n_sig = n_all = 0
    for key, d in L.items():
        x = window_inputs(d["live_words"]).astype(np.float32)
        p_emu = np.asarray(hm.predict(np.ascontiguousarray(x))).reshape(-1)
        p_chip = (np.asarray(d["live_rec"])[:, 6] & 0x3FFFF) / 1024.0
        n_sig += int((p_chip == p_emu).sum()); n_all += len(p_emu)
    print(f"sigmoid output identical to the fixed-point model: {n_sig:,}/{n_all:,} shots")

## 4. Optional: capture new shots on a board

This replays a few test traces, reads NN_live's record and the window it classified, and runs the same checks. The
program triggers readout 0 (QICK's FIR path), and together readout 1 (the boxcar stream) and NN_live's classify trigger
`NN_ADV` earlier. Skip this section without a board.

In [ ]:
RUN_ON_BOARD = False            # set True to capture
BOARD_HOST, NS_PORT, PROXY_NAME = "10.1.9.26", 8888, "myqick"
CAL_DIR = Path("outputs/calibration")        # notebook 01's k and c
TRACES_X, TRACES_Y = "data/test20k_real_X.npy", "data/test20k_real_y.npy"
FIRST_ROW, N_SHOTS = 2000, 100
NN_ADV = 0.1135

if RUN_ON_BOARD:
    import Pyro4
    from qick import QickConfig
    from qick.asm_v2 import AveragerProgramV2
    Pyro4.config.SERIALIZER = "pickle"; Pyro4.config.PICKLE_PROTOCOL_VERSION = 4
    soc = Pyro4.Proxy(Pyro4.locateNS(host=BOARD_HOST, port=NS_PORT).lookup(PROXY_NAME))
    soccfg = QickConfig(soc.get_cfg())
    soccfg["tprocs"][0]["output_pins"].append(("trig", 11, 0, "NN classify trigger (trig_11_o)"))
    CLS_PIN = len(soccfg["tprocs"][0]["output_pins"]) - 1
    print("boxcar path:", soc.box_config(sel=1, shift=7, m=11))

    class LiveShot(AveragerProgramV2):
        def _initialize(self, cf):
            self.declare_gen(ch=0, nqz=1)
            for ch in (0, 1):
                self.declare_readout(ch=ch, length=cf["readout_length"])
                self.add_readoutconfig(ch=ch, name=f"ro{ch}", freq=cf["freq"], gen_ch=0)
            self.add_envelope(ch=0, name="env", idata=cf["waveform_I"], qdata=cf["waveform_Q"])
            self.add_pulse(ch=0, name="p", style="arb", envelope="env", gain=1, freq=cf["freq"], phase=0.0,
                           outsel="product", mode="oneshot", stdysel="zero", ro_ch=0)

        def _body(self, cf):
            self.pulse(ch=0, name="p")
            self.send_readoutconfig(ch=0, name="ro0", t=0); self.send_readoutconfig(ch=1, name="ro1", t=0)
            self.trigger(ros=[0], t=cf["adc_trig_offset"], width=10)
            self.trigger(ros=[1], pins=[CLS_PIN], t=cf["adc_trig_offset"] - cf["nn_adv"], width=10)

    k = complex(np.load(CAL_DIR / f"k_{BOARD_HOST}.npy")); c = complex(np.load(CAL_DIR / f"c_{BOARD_HOST}.npy"))
    Xall = np.load(DATA_DIR / TRACES_X, mmap_mode="r"); yall = np.load(DATA_DIR / TRACES_Y).astype(int)
    zr = (np.asarray(Xall[:, 0::2]) + 1j * np.asarray(Xall[:, 1::2]) - c) / k
    scale = 32766 * 0.99 / (max(np.abs(zr.real).max(), np.abs(zr.imag).max()) * 1.0001); del zr
    maxv = soccfg["gens"][0]["maxv"]
    xo, xn = np.linspace(0, 1, 770), np.linspace(0, 1, 11000)
    words, recs, rows = [], [], np.arange(FIRST_ROW, FIRST_ROW + N_SHOTS)
    for r in rows:
        z = (np.asarray(Xall[r, 0::2]) + 1j * np.asarray(Xall[r, 1::2]) - c) / k
        w = np.empty(22000); w[0::2] = np.interp(xn, xo, z.real) * scale; w[1::2] = np.interp(xn, xo, z.imag) * scale
        w = np.pad(w, (0, (-w.size) % 32))                  # envelopes must be a multiple of 16 I/Q pairs
        w = np.clip(w, -maxv, maxv)
        cf = dict(readout_length=1.393, adc_trig_offset=0.31, nn_adv=NN_ADV, freq=50.0, waveform_I=w[0::2], waveform_Q=w[1::2])
        LiveShot(soccfg, reps=1, final_delay=100, cfg=cf).acquire_decimated(soc, rounds=1, progress=False)
        lv = soc.live_latest()
        recs.append(lv[3:]); words.append(soc.live_read_block(16, WINDOW))
    words, recs, y = np.array(words, np.uint32), np.array(recs, np.int64), yall[rows]
    ok_s, ok_r, x = check_sums_and_rule(words, recs)
    show("NN_live, new shots", fstats((recs[:, 7] & 1).astype(int), y))
    print(f"first-layer sums exact {ok_s.sum()}/{len(y)}; decision rule {ok_r.sum()}/{len(y)}")
    if CHECK_SIGMOID:
        p_emu = np.asarray(hm.predict(np.ascontiguousarray(x.astype(np.float32)))).reshape(-1)
        print(f"sigmoid identical to the fixed-point model {int(((recs[:, 6] & 0x3FFFF) / 1024.0 == p_emu).sum())}/{len(y)}")

**Notes**
- NN_axi's per-shot probabilities come from reading its result BRAM after each shot; the repository's
  `analysis_scripts/diagnostics/score_armed_fidelity.py` does that acquisition. Its bit-exact check against the fixed-point
  model of the 770-sample network uses the window QICK's buffer recorded (`dec_iq` in the NN_axi files) and the method of
  notebook 04.
- With the full campaign data, the fidelities above are the paper's Table II numbers. With a subset, expect wider
  intervals.